# 📊 Notebook 00 — Synthetic Data Generator
## Financial Representation Learning System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cliffordnwanna/FRL-SYSTEM/blob/main/notebooks/00_synthetic_data_generator.ipynb)

**Purpose:** Generate a realistic synthetic financial dataset that mirrors the structure of real banking/fintech data — without using any real customer information.

**Output:** 6 CSV files saved to `data/synthetic/`
- `customers.csv` — 5,000 customers with 6 behavioural archetypes  
- `transactions.csv` — ~180,000 transaction records
- `app_events.csv` — ~120,000 digital platform events
- `counterparties.csv` — 2,500 merchants, utilities, individuals
- `products.csv` — product holdings per customer
- `complaints.csv` — complaint records

**Runtime:** ~2 minutes (no GPU required)

---

## Step 1 — Environment Setup

In [ ]:
# Install dependencies (Colab environment)
import subprocess, sys

packages = ["torch", "pandas", "numpy", "tqdm", "scikit-learn", "matplotlib"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ Dependencies ready")

In [ ]:
# Clone the repository (if running on Colab)
import os

REPO_URL = "https://github.com/cliffordnwanna/FRL-SYSTEM.git"
REPO_DIR = "/content/FRL-SYSTEM"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"✅ Repo cloned to {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print(f"✅ Repo updated")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Mount Google Drive and restore artifacts from earlier notebooks/sessions
# This makes the pipeline resumable across disconnects, closed tabs, or a
# fresh runtime the next day — you don't need to keep one Colab tab open.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/FRL-SYSTEM-artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("data/synthetic", exist_ok=True)

restored = 0
for fname in os.listdir(DRIVE_DIR):
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join("data/synthetic", fname)
    if os.path.isfile(src):
        shutil.copy(src, dst)
        restored += 1
        print(f"Restored {fname} from Drive")
print(f"✅ Restored {restored} artifact(s) from Drive" if restored else "ℹ️ No prior artifacts found in Drive yet")

## Step 2 — Generate the Data

In [ ]:
import sys
sys.path.insert(0, ".")

from src.data_generator import generate_all
from pathlib import Path

# Generate all tables
dfs = generate_all(output_dir=Path("data/synthetic"))

## Step 3 — Explore the Data

In [ ]:
import pandas as pd

customers = pd.read_csv("data/synthetic/customers.csv")
transactions = pd.read_csv("data/synthetic/transactions.csv")
app_events = pd.read_csv("data/synthetic/app_events.csv")
counterparties = pd.read_csv("data/synthetic/counterparties.csv")
products = pd.read_csv("data/synthetic/products.csv")
complaints = pd.read_csv("data/synthetic/complaints.csv")

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
for name, df in [("customers", customers), ("transactions", transactions),
                  ("app_events", app_events), ("counterparties", counterparties),
                  ("products", products), ("complaints", complaints)]:
    print(f"  {name:20s}: {len(df):>8,} rows | {len(df.columns)} cols")
print("=" * 50)

In [ ]:
# Customer archetype distribution
print("\nCustomer Archetype Distribution:")
print(customers['archetype'].value_counts())
print("\nAEPD Stage Distribution (Ground Truth):")
print(customers['aepd_stage'].value_counts())

In [ ]:
# Transaction channel mix
print("\nTransaction Channel Mix:")
print(transactions['channel'].value_counts())
print("\nTransaction Type:")
print(transactions['txn_type'].value_counts())

In [ ]:
# Amount distribution
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Synthetic Dataset Overview", fontsize=14, fontweight="bold")

# Amount distribution (log scale)
axes[0].hist(np.log10(transactions['amount'].clip(lower=1)), bins=50, color='#3498DB', edgecolor='white', linewidth=0.5)
axes[0].set_title("Transaction Amount Distribution (log₁₀ scale)")
axes[0].set_xlabel("log₁₀(Amount)")
axes[0].set_ylabel("Count")

# Events per customer
txn_per_cust = transactions.groupby('customer_id').size()
axes[1].hist(txn_per_cust, bins=50, color='#2ECC71', edgecolor='white', linewidth=0.5)
axes[1].set_title("Transactions per Customer")
axes[1].set_xlabel("Transaction Count")

# AEPD stage pie
aepd_counts = customers['aepd_stage'].value_counts()
colors = {'A': '#E74C3C', 'E': '#F39C12', 'P': '#2ECC71', 'D': '#3498DB'}
axes[2].pie(aepd_counts.values, labels=aepd_counts.index,
            colors=[colors.get(s, '#999') for s in aepd_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[2].set_title("AEPD Stage Distribution (Ground Truth)")

plt.tight_layout()
plt.savefig("docs/results/00_dataset_overview.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Plot saved to docs/results/00_dataset_overview.png")

In [ ]:
# Preview each table
print("customers.csv — first 3 rows:")
display(customers.head(3))

print("\ntransactions.csv — first 3 rows:")
display(transactions.head(3))

print("\napp_events.csv — first 3 rows:")
display(app_events.head(3))

In [ ]:
# Persist this notebook's outputs to Drive so the next notebook — in this
# session or a brand new one tomorrow — can pick up where this left off.
import shutil, os
DRIVE_DIR = "/content/drive/MyDrive/FRL-SYSTEM-artifacts"
saved = 0
for fname in os.listdir("data/synthetic"):
    if fname.endswith((".csv", ".pkl", ".pt")):
        shutil.copy(os.path.join("data/synthetic", fname),
                    os.path.join(DRIVE_DIR, fname))
        saved += 1
print(f"✅ Saved {saved} artifact(s) to Drive: {DRIVE_DIR}")

## ✅ Notebook 00 Complete

**What was generated:**
- 5,000 synthetic customers across 6 behavioural archetypes
- Ground truth AEPD labels for validation in Notebook 05
- Realistic transaction amounts (log-normal distribution)
- App engagement events per digital channel

**Next:** Run `01_event_tokenizer.ipynb` to convert these events into the financial vocabulary sequences that feed the encoder.

---
*Data is saved to `data/synthetic/`. These files are gitignored (too large for GitHub). Re-run this notebook to regenerate them.*